In [49]:
import pandas as pd
import re
from collections import Counter

# 1. 데이터 로드 및 전처리
df_ddgemb = pd.read_csv(r'C:\Users\Kunny\Documents\GitHub\CAGI\EvoStructCLIP\S669\S669.tsv', sep='\t')
df_zenodo = pd.read_csv(r'C:\Users\Kunny\Documents\GitHub\CAGI\EvoStructCLIP\S669\S669_Zenodo\S669.csv', sep=',')

def extract_num(mut_str):
    match = re.search(r'\d+', str(mut_str))
    return int(match.group()) if match else None

# CSV 미리 정리 (매칭 속도 향상)
df_zenodo['WT'] = df_zenodo['PDB_Mut'].str[0]
df_zenodo['MT'] = df_zenodo['PDB_Mut'].str[-1]
df_zenodo['PDB_POS_VAL'] = df_zenodo['PDB_Mut'].apply(extract_num)

df_ddgemb['PDB_ID_FINAL'] = None
df_ddgemb['PDB_POS_FINAL'] = None
df_ddgemb['OFFSET'] = None

# 2. 단백질별로 루프 실행
for protein_id in df_ddgemb['PDB'].unique():
    subset_tsv = df_ddgemb[df_ddgemb['PDB'] == protein_id]
    
    # 해당 단백질에 대응하는 CSV 데이터 찾기 (예: P47992 -> 1j8iA 등)
    # 여기서는 간단히 WT, MT, DDG가 겹치는 데이터가 있는 CSV 그룹을 찾습니다.
    # (실제로는 앞서 만든 mapping_results.json의 PDB ID를 활용하면 더 정확합니다)
    
    # 2-1. 먼저 해당 단백질의 가능한 모든 Offset 후보 계산
    offsets = []
    for _, t_row in subset_tsv.iterrows():
        matches = df_zenodo[
            (df_zenodo['WT'] == t_row['WT']) & 
            (df_zenodo['MT'] == t_row['MT']) & 
            (abs(df_zenodo['Experimental_DDG_dir'] - t_row['DDG']) < 0.001) # 소수점 오차 허용
        ]
        for _, c_row in matches.iterrows():
            offsets.append(t_row['POS'] - c_row['PDB_POS_VAL'])
    
    if not offsets:
        continue
        
    # 2-2. 가장 지배적인 Offset(최빈값) 결정
    common_offset = Counter(offsets).most_common(1)[0][0]
    
    # 2-3. 결정된 Offset을 바탕으로 정밀 매핑
    for idx in subset_tsv.index:
        t_row = df_ddgemb.loc[idx]
        # WT, MT, DDG가 맞으면서 Offset까지 완벽한 것 찾기
        final_match = df_zenodo[
            (df_zenodo['WT'] == t_row['WT']) & 
            (df_zenodo['MT'] == t_row['MT']) & 
            (abs(df_zenodo['Experimental_DDG_dir'] - t_row['DDG']) < 0.01) &
            ((t_row['POS'] - df_zenodo['PDB_POS_VAL']) == common_offset)
        ]
        
        if len(final_match) >= 1:
            df_ddgemb.at[idx, 'PDB_ID_FINAL'] = final_match.iloc[0]['Protein']
            df_ddgemb.at[idx, 'PDB_POS_FINAL'] = final_match.iloc[0]['PDB_POS_VAL']
            df_ddgemb.at[idx, 'OFFSET'] = common_offset
        else:
            print(f"⚠️ 매치 실패: {protein_id} POS {t_row['POS']} (일치하는 Offset {common_offset} 없음)")

# 3. 결과 확인
print("\n[매핑 결과 예시 - P47992]")
print(df_ddgemb[df_ddgemb['PDB'] == 'P47992'][['PDB', 'WT', 'POS', 'PDB_POS_FINAL', 'OFFSET']].head(10))

df_ddgemb.to_csv("S669_Offset_Corrected.tsv", sep='\t', index=False)

⚠️ 매치 실패: P78352 POS 332 (일치하는 Offset 0 없음)
⚠️ 매치 실패: P78352 POS 332 (일치하는 Offset 0 없음)

[매핑 결과 예시 - P47992]
        PDB WT  POS PDB_POS_FINAL OFFSET
191  P47992  K   46            25     21
192  P47992  K   87            66     21
193  P47992  R   44            23     21
194  P47992  R   56            35     21
195  P47992  R   64            43     21
196  P47992  R   30             9     21


In [50]:
import pandas as pd
import re
df = pd.read_csv(r'C:\Users\Kunny\Documents\GitHub\CAGI\EvoStructCLIP\S669\S669_Offset_Corrected.tsv', sep='\t')

In [51]:
df

,PDB,WT,POS,MT,DDG,PDB_ID_FINAL,PDB_POS_FINAL,OFFSET
0,P0A9D2,S,11,A,-1.800,1a0fA,11,0
1,P00149,A,125,H,-2.690,1a7vA,104,21
2,P00149,A,87,H,-1.980,1a7vA,66,21
3,P00149,A,112,H,-1.700,1a7vA,91,21
4,P00149,D,24,H,-1.360,1a7vA,3,21
...,...,...,...,...,...,...,...,...
664,1fh5H,S,15,N,0.600,1fh5H,18,-3
665,1iv7B,A,73,C,0.500,1iv7B,173,-100
666,1iv7B,F,89,C,-2.500,1iv7B,189,-100
667,1iv7B,L,62,C,-1.900,1iv7B,162,-100


In [52]:
df = df[["PDB",
           "WT",
           "POS",
           "MT",
           "DDG",
           "PDB_ID_FINAL",
           "PDB_POS_FINAL",
           ]]

In [53]:
df

,PDB,WT,POS,MT,DDG,PDB_ID_FINAL,PDB_POS_FINAL
0,P0A9D2,S,11,A,-1.800,1a0fA,11
1,P00149,A,125,H,-2.690,1a7vA,104
2,P00149,A,87,H,-1.980,1a7vA,66
3,P00149,A,112,H,-1.700,1a7vA,91
4,P00149,D,24,H,-1.360,1a7vA,3
...,...,...,...,...,...,...,...
664,1fh5H,S,15,N,0.600,1fh5H,18
665,1iv7B,A,73,C,0.500,1iv7B,173
666,1iv7B,F,89,C,-2.500,1iv7B,189
667,1iv7B,L,62,C,-1.900,1iv7B,162


In [54]:
df["PDB_ID"] = df["PDB_ID_FINAL"].str[:-1]
df["Chain"] = df["PDB_ID_FINAL"].str[-1]

In [55]:
df

,PDB,WT,POS,MT,DDG,PDB_ID_FINAL,PDB_POS_FINAL,PDB_ID,Chain
0,P0A9D2,S,11,A,-1.800,1a0fA,11,1a0f,A
1,P00149,A,125,H,-2.690,1a7vA,104,1a7v,A
2,P00149,A,87,H,-1.980,1a7vA,66,1a7v,A
3,P00149,A,112,H,-1.700,1a7vA,91,1a7v,A
4,P00149,D,24,H,-1.360,1a7vA,3,1a7v,A
...,...,...,...,...,...,...,...,...,...
664,1fh5H,S,15,N,0.600,1fh5H,18,1fh5,H
665,1iv7B,A,73,C,0.500,1iv7B,173,1iv7,B
666,1iv7B,F,89,C,-2.500,1iv7B,189,1iv7,B
667,1iv7B,L,62,C,-1.900,1iv7B,162,1iv7,B


In [56]:
df = df.drop(columns=['PDB_ID_FINAL'])

df = df.rename(columns={
    "MT": "Mut",
    "POS": "SeqPos",
    "PDB": "Key",
    "PDB_POS_FINAL": "MutPos"
})

df

,Key,WT,SeqPos,Mut,DDG,MutPos,PDB_ID,Chain
0,P0A9D2,S,11,A,-1.800,11,1a0f,A
1,P00149,A,125,H,-2.690,104,1a7v,A
2,P00149,A,87,H,-1.980,66,1a7v,A
3,P00149,A,112,H,-1.700,91,1a7v,A
4,P00149,D,24,H,-1.360,3,1a7v,A
...,...,...,...,...,...,...,...,...
664,1fh5H,S,15,N,0.600,18,1fh5,H
665,1iv7B,A,73,C,0.500,173,1iv7,B
666,1iv7B,F,89,C,-2.500,189,1iv7,B
667,1iv7B,L,62,C,-1.900,162,1iv7,B


In [57]:
df["FILE_PATH"] = df["PDB_ID"] + ".pdb"

In [58]:
df.to_csv("S669_processed.tsv", sep='\t', index=False)

In [60]:
df

,Key,WT,SeqPos,Mut,DDG,MutPos,PDB_ID,Chain,FILE_PATH
0,P0A9D2,S,11,A,-1.800,11,1a0f,A,1a0f.pdb
1,P00149,A,125,H,-2.690,104,1a7v,A,1a7v.pdb
2,P00149,A,87,H,-1.980,66,1a7v,A,1a7v.pdb
3,P00149,A,112,H,-1.700,91,1a7v,A,1a7v.pdb
4,P00149,D,24,H,-1.360,3,1a7v,A,1a7v.pdb
...,...,...,...,...,...,...,...,...,...
664,1fh5H,S,15,N,0.600,18,1fh5,H,1fh5.pdb
665,1iv7B,A,73,C,0.500,173,1iv7,B,1iv7.pdb
666,1iv7B,F,89,C,-2.500,189,1iv7,B,1iv7.pdb
667,1iv7B,L,62,C,-1.900,162,1iv7,B,1iv7.pdb
